In [ ]:
```xml
<VSCode.Cell language="markdown">
# 🚦 Advanced API Rate Limiting Guide

**Distributed rate limiting algorithms, token bucket, sliding window, and adaptive throttling**

This guide covers:
- Rate limiting algorithms
- Token bucket implementation
- Sliding window log
- Distributed rate limiting
- Adaptive throttling
- Client-side rate limiting
- Quota management
- Priority-based rate limiting
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 1. Rate Limiting Algorithms

### Token Bucket Implementation
```python
from dataclasses import dataclass
from typing import Dict, Optional, Tuple
from datetime import datetime, timedelta
from enum import Enum
import time

class RateLimitStatus(Enum):
    ALLOWED = "allowed"
    RATE_LIMITED = "rate_limited"
    QUOTA_EXCEEDED = "quota_exceeded"

@dataclass
class RateLimitConfig:
    """Rate limit configuration"""
    max_requests: int
    window_size_seconds: int
    burst_size: int = None

class TokenBucket:
    """Token bucket rate limiter"""
    
    def __init__(self, capacity: int, refill_rate: float):
        self.capacity = capacity
        self.refill_rate = refill_rate  # tokens per second
        self.tokens = capacity
        self.last_refill_time = datetime.utcnow()
    
    def allow_request(self, tokens_required: int = 1) -> Tuple[bool, int]:
        """Check if request allowed"""
        
        self._refill()
        
        if self.tokens >= tokens_required:
            self.tokens -= tokens_required
            return True, int(self.tokens)
        
        return False, 0
    
    def _refill(self):
        """Refill tokens"""
        
        now = datetime.utcnow()
        time_passed = (now - self.last_refill_time).total_seconds()
        
        tokens_to_add = time_passed * self.refill_rate
        self.tokens = min(self.capacity, self.tokens + tokens_to_add)
        
        self.last_refill_time = now
    
    def get_retry_after(self) -> int:
        """Get seconds until next request allowed"""
        
        if self.tokens >= 1:
            return 0
        
        tokens_needed = 1
        seconds_needed = tokens_needed / self.refill_rate
        
        return int(seconds_needed) + 1

class SlidingWindowLog:
    """Sliding window log rate limiter"""
    
    def __init__(self, max_requests: int, window_seconds: int):
        self.max_requests = max_requests
        self.window_seconds = window_seconds
        self.request_log: Dict[str, list] = {}
    
    def allow_request(self, client_id: str) -> Tuple[bool, int]:
        """Check if request allowed"""
        
        now = datetime.utcnow()
        window_start = now - timedelta(seconds=self.window_seconds)
        
        # Get or create log for client
        if client_id not in self.request_log:
            self.request_log[client_id] = []
        
        # Remove old requests outside window
        self.request_log[client_id] = [
            ts for ts in self.request_log[client_id] 
            if ts > window_start
        ]
        
        # Check if allowed
        if len(self.request_log[client_id]) < self.max_requests:
            self.request_log[client_id].append(now)
            return True, self.max_requests - len(self.request_log[client_id])
        
        return False, 0
    
    def get_reset_time(self, client_id: str) -> Optional[datetime]:
        """Get when quota resets"""
        
        if client_id not in self.request_log or not self.request_log[client_id]:
            return None
        
        oldest_request = min(self.request_log[client_id])
        reset_time = oldest_request + timedelta(seconds=self.window_seconds)
        
        return reset_time

class SlidingWindowCounter:
    """Sliding window counter rate limiter"""
    
    def __init__(self, max_requests: int, window_seconds: int):
        self.max_requests = max_requests
        self.window_seconds = window_seconds
        self.counter: Dict[str, Dict] = {}
    
    def allow_request(self, client_id: str) -> Tuple[bool, int]:
        """Check if request allowed"""
        
        now = datetime.utcnow()
        current_minute = now.replace(second=0, microsecond=0)
        
        if client_id not in self.counter:
            self.counter[client_id] = {}
        
        # Count requests in current and previous window
        prev_window = (current_minute - timedelta(seconds=self.window_seconds)).isoformat()
        current_window = current_minute.isoformat()
        
        if current_window not in self.counter[client_id]:
            self.counter[client_id][current_window] = 0
        
        current_count = self.counter[client_id].get(current_window, 0)
        prev_count = self.counter[client_id].get(prev_window, 0)
        
        # Calculate weighted count
        time_passed = (now - current_minute).total_seconds()
        weight = 1 - (time_passed / self.window_seconds)
        
        weighted_count = current_count + (prev_count * weight)
        
        if weighted_count < self.max_requests:
            self.counter[client_id][current_window] += 1
            return True, int(self.max_requests - weighted_count)
        
        return False, 0
```
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 2. Distributed Rate Limiting

### Distributed Implementation
```python
class DistributedRateLimiter:
    """Distributed rate limiter using Redis-like backend"""
    
    def __init__(self, redis_client=None):
        self.redis = redis_client
        self.local_cache: Dict = {}
    
    def check_rate_limit(self, client_id: str, config: RateLimitConfig) -> Tuple[bool, Dict]:
        """Check rate limit with distributed consistency"""
        
        key = f"rate_limit:{client_id}"
        
        # Try distributed backend first
        if self.redis:
            result = self._check_distributed(key, config)
        else:
            result = self._check_local(key, config)
        
        return result
    
    def _check_distributed(self, key: str, config: RateLimitConfig) -> Tuple[bool, Dict]:
        """Check against Redis"""
        
        current_count = self.redis.get(key) or 0
        
        if current_count < config.max_requests:
            self.redis.incr(key)
            self.redis.expire(key, config.window_size_seconds)
            
            return True, {
                'remaining': config.max_requests - current_count - 1,
                'reset_in': self.redis.ttl(key)
            }
        
        return False, {
            'remaining': 0,
            'reset_in': self.redis.ttl(key)
        }
    
    def _check_local(self, key: str, config: RateLimitConfig) -> Tuple[bool, Dict]:
        """Check against local cache"""
        
        now = datetime.utcnow()
        
        if key not in self.local_cache:
            self.local_cache[key] = {
                'count': 0,
                'reset_time': now + timedelta(seconds=config.window_size_seconds)
            }
        
        bucket = self.local_cache[key]
        
        # Reset if window expired
        if now > bucket['reset_time']:
            bucket['count'] = 0
            bucket['reset_time'] = now + timedelta(seconds=config.window_size_seconds)
        
        if bucket['count'] < config.max_requests:
            bucket['count'] += 1
            
            reset_in = (bucket['reset_time'] - now).total_seconds()
            
            return True, {
                'remaining': config.max_requests - bucket['count'],
                'reset_in': int(reset_in)
            }
        
        return False, {
            'remaining': 0,
            'reset_in': int((bucket['reset_time'] - now).total_seconds())
        }

class AdaptiveThrottling:
    """Adaptive rate limiting based on system load"""
    
    def __init__(self, base_rate: int):
        self.base_rate = base_rate
        self.current_rate = base_rate
        self.system_load_history: list = []
    
    def update_rate(self, cpu_usage: float, memory_usage: float):
        """Adjust rate based on system load"""
        
        load = (cpu_usage + memory_usage) / 2
        self.system_load_history.append(load)
        
        # Keep last 10 samples
        if len(self.system_load_history) > 10:
            self.system_load_history.pop(0)
        
        avg_load = sum(self.system_load_history) / len(self.system_load_history)
        
        # Adjust rate
        if avg_load > 80:
            # High load - reduce rate by 20%
            self.current_rate = int(self.base_rate * 0.8)
        elif avg_load > 60:
            # Medium load - reduce rate by 10%
            self.current_rate = int(self.base_rate * 0.9)
        else:
            # Low load - restore full rate
            self.current_rate = self.base_rate
    
    def get_current_rate(self) -> int:
        """Get current rate limit"""
        return self.current_rate
```
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 3. Client-Side & Priority Rate Limiting

### Client Priority System
```python
class ClientPriority(Enum):
    FREE = 1
    STANDARD = 2
    PREMIUM = 3
    ENTERPRISE = 4

class PriorityBasedRateLimiter:
    """Rate limit with priority tiers"""
    
    def __init__(self):
        self.client_tiers: Dict[str, ClientPriority] = {}
        self.rate_limits: Dict[ClientPriority, RateLimitConfig] = {
            ClientPriority.FREE: RateLimitConfig(100, 3600),
            ClientPriority.STANDARD: RateLimitConfig(1000, 3600),
            ClientPriority.PREMIUM: RateLimitConfig(10000, 3600),
            ClientPriority.ENTERPRISE: RateLimitConfig(100000, 3600),
        }
        self.limiters: Dict[str, TokenBucket] = {}
    
    def register_client(self, client_id: str, tier: ClientPriority):
        """Register client with tier"""
        
        self.client_tiers[client_id] = tier
        
        # Create limiter for client
        config = self.rate_limits[tier]
        refill_rate = config.max_requests / config.window_size_seconds
        burst = config.burst_size or config.max_requests
        
        self.limiters[client_id] = TokenBucket(burst, refill_rate)
    
    def check_rate_limit(self, client_id: str) -> Tuple[bool, Dict]:
        """Check rate limit for client"""
        
        if client_id not in self.limiters:
            return False, {'error': 'client_not_registered'}
        
        allowed, remaining = self.limiters[client_id].allow_request()
        tier = self.client_tiers[client_id]
        
        return allowed, {
            'tier': tier.name,
            'remaining': remaining,
            'retry_after': self.limiters[client_id].get_retry_after() if not allowed else 0
        }

class ClientSideRateLimiter:
    """Client-side rate limiting"""
    
    def __init__(self, max_requests: int, window_seconds: int):
        self.max_requests = max_requests
        self.window_seconds = window_seconds
        self.request_times: list = []
    
    def is_allowed(self) -> bool:
        """Check if request allowed"""
        
        now = time.time()
        window_start = now - self.window_seconds
        
        # Remove old requests
        self.request_times = [t for t in self.request_times if t > window_start]
        
        # Check limit
        if len(self.request_times) < self.max_requests:
            self.request_times.append(now)
            return True
        
        return False
    
    def wait_if_needed(self):
        """Wait if rate limited"""
        
        if not self.is_allowed():
            oldest = min(self.request_times)
            sleep_time = max(0, (oldest + self.window_seconds) - time.time())
            time.sleep(sleep_time + 0.1)
```
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 4. Rate Limiting Checklist

✅ **Algorithm Selection**
- [ ] Requirements defined
- [ ] Algorithm chosen
- [ ] Accuracy verified
- [ ] Performance tested
- [ ] Scaling validated

✅ **Implementation**
- [ ] Limiter deployed
- [ ] Distributed setup
- [ ] Fallbacks configured
- [ ] Monitoring active
- [ ] Alerts defined

✅ **Client Tiers**
- [ ] Tiers defined
- [ ] Limits configured
- [ ] Enforcement active
- [ ] Documentation clear
- [ ] Support ready

✅ **Monitoring**
- [ ] Metrics tracked
- [ ] Anomalies detected
- [ ] Performance optimal
- [ ] Costs managed
- [ ] Reports generated

✅ **Operational**
- [ ] Documentation
- [ ] Team trained
- [ ] SLAs defined
- [ ] Incident response
- [ ] Continuous improvement
</VSCode.Cell>
```